# Capstone Phase 5 上机：商业模式与价值评估

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 核心任务

为 AI 营销 Agent SaaS「MarketingAgent Pro」构建完整投资评估，整合 Phase 4 因果效果：
1. 商业模式画布（9宫格结构化，价值主张整合Phase 4 ATE）
2. ATE -> ARPU -> DCF 估值（NPV / IRR / 回收期 / PI）
3. 蒙特卡洛模拟（传播ATE置信区间不确定性，估值分布 + 概率分析）
4. 敏感性分析（龙卷风图，含ATE和推理成本）
5. 天道推演多路径场景分析（Bull/Base/Bear，ATE CI上下界为边界）

**真实库**：numpy-financial（NPV/IRR）｜ scipy.stats（蒙特卡洛）｜ pandas + matplotlib
**真实数据**：HubSpot 2023 财报 + Jasper AI Crunchbase + OpenAI API定价 + Phase 4因果效果（ATE）


## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：

> 需要 numpy-financial, scipy, pandas, matplotlib。通常已随 conda/venv 安装。
> numpy-financial 提供 NPV/IRR 等标准金融函数。


In [ ]:
# !pip install numpy-financial scipy pandas matplotlib -q
import numpy as np
import pandas as pd
import numpy_financial as npf
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("环境就绪: numpy-financial + scipy.stats + pandas + matplotlib")


## 1. 商业模式画布（Business Model Canvas）

AI 商业模式画布在传统九宫格基础上适配 AI 原生特征：
- **收入流**：新增 outcome-based pricing（alpha=3.33%价值捕获率）
- **核心资源**：新增数据资产 + AI 模型 + 算力 + 因果实验数据
- **核心活动**：新增模型训练/评估 + Agent运维 + 因果实验
- **成本结构**：新增推理成本（持续运营成本，30%）
- **核心伙伴**：新增MCP协议连接 + A2A协作

**案例**：MarketingAgent Pro - AI 原生营销 Agent 平台
- Phase 4整合：ATE=+3.8pp转化率提升（95% CI: [2.2pp, 5.4pp]）作为价值主张核心
- 数据校准：HubSpot 2023 财报（gross margin ~78%）、Jasper AI（$125M ARR）


In [ ]:
# 1. 商业模式画布（9宫格）
# Phase 4 ATE=3.8pp 转化率提升 -> 价值主张核心
canvas_data = {
    '构件': ['客户细分', '价值主张', '渠道', '客户关系', '收入流',
            '核心资源', '核心活动', '核心伙伴', '成本结构'],
    'MarketingAgent Pro': [
        '中型企业营销部门(50-500人, 年预算100-1000万)',
        'Phase4验证: ATE=+3.8pp转化率提升(95%CI[2.2,5.4]) -> 可量化ROI',
        '直销(大客户) + 自助注册(中小) + Agent市场分发',
        'AI驱动个性化onboarding + Customer Success Agent',
        '基础订阅$500/月 + 按转化效果付费(outcome-based, alpha=3.33%)',
        '营销专有数据 + 因果实验数据 + Agent编排 + 行业Know-how',
        'Agent训练/优化 + 因果实验运维 + 数据管道 + 效果评估',
        '基础模型(OpenAI/Anthropic) + 广告平台API + 数据伙伴 + MCP协议',
        '推理成本30% + 数据15% + 人才30% + 营销15% + 合规10%'
    ],
    '传统SaaS对比': [
        '同上(AI未改变目标客户)',
        '工具辅助(无因果验证, 效果不可量化)',
        '直销+自助注册(无Agent渠道)',
        '人工CSM + 工单系统',
        'Seat-based($X/用户/月, 无效果分成)',
        '软件代码 + 客户名单 + 品牌',
        '软件开发 + 客户成功 + 销售',
        '云服务 + 支付 + CDN',
        '研发40% + 销售30% + 运营20% + 合规10%'
    ]
}
canvas_df = pd.DataFrame(canvas_data)
print("=== MarketingAgent Pro 商业模式画布 (Capstone Phase 5) ===")
print(canvas_df.to_string(index=False))


## 2. Phase 4 ATE -> ARPU 推导 + DCF 估值模型

**Capstone核心整合**：将Phase 4因果效果（ATE）转化为商业价值（ARPU），再通过DCF模型评估投资可行性。

推导链：ATE -> 月增转化 -> 月增收 -> ARPU -> DCF -> NPV

| 参数 | 值 | 来源 |
|------|-----|------|
| ATE | 0.038 (3.8pp) | Phase 4 因果推断（DoWhy） |
| 月触达 | 10,000 | 营销Agent系统基准 |
| AOV | $158 | 行业平均订单价值 |
| 价值捕获率 | 3.33% | outcome-based pricing (alpha) |
| 初始投资 | $2,000K | 开发团队 + GTM 投入 |
| 毛利率 | 65% | 含推理成本 30% + 数据 5% |
| 折现率 | 15% | VC 典型 SaaS 要求回报 |
| 评估窗口 | 5年 | J 曲线效应需 3-5 年 |

**numpy-financial 核心函数**：
- `npf.npv(rate, cashflows)` - 净现值
- `npf.irr(cashflows)` - 内部收益率


In [ ]:
# 2. Phase 4 ATE -> ARPU 推导 + DCF 5年模型 + NPV
# Capstone核心整合: 因果效果(ATE) -> 商业价值(ARPU) -> 投资评估(NPV)

# Phase 4 因果效果参数
ate = 0.038            # ATE: 3.8pp转化率提升 (95% CI: [0.022, 0.054])
monthly_reach = 10000  # 每客户月均触达prospect数
aov = 158              # 平均订单价值 ($)
capture_rate = 0.0333  # 价值捕获率 (outcome-based, alpha=3.33%)

# ARPU推导: 触达 * ATE * AOV * 捕获率 = 月收入
arpu_monthly = monthly_reach * ate * aov * capture_rate
arpu_annual = arpu_monthly * 12
arpu_annual_k = arpu_annual / 1000

print("=== Phase 4 ATE -> Phase 5 ARPU 推导链 ===")
print(f"  ATE (Phase 4):           {ate:.3f} (+{ate*100:.1f}pp转化率提升)")
print(f"  月触达:                  {monthly_reach:,} prospects")
print(f"  月增转化:                {monthly_reach * ate:.0f} 次")
print(f"  平均订单价值:            ${aov}")
print(f"  月增收:                  ${monthly_reach * ate * aov:,.0f}")
print(f"  价值捕获率:              {capture_rate:.1%}")
print(f"  ARPU (月):               ${arpu_monthly:,.0f}")
print(f"  ARPU (年):               ${arpu_annual:,.0f} = ${arpu_annual_k:.1f}K")

# DCF 5年模型
initial_investment = 2000  # $K
discount_rate = 0.15
customers = [0, 30, 80, 160, 260, 380]
gross_margin = 0.65  # 推理成本30% + 数据5%
opex = [0, 800, 1200, 1800, 2500, 3200]
years = list(range(6))

revenue = [c * arpu_annual_k for c in customers]
gross_profit = [r * gross_margin for r in revenue]
fcf = [gp - op for gp, op in zip(gross_profit, opex)]
fcf[0] = -initial_investment

dcf_df = pd.DataFrame({
    'Year': years,
    'Customers': customers,
    'ARPU($K)': [round(arpu_annual_k, 1)] * 6,
    'Revenue($K)': [round(r, 1) for r in revenue],
    'GrossProfit($K)': [round(gp, 1) for gp in gross_profit],
    'OpEx($K)': opex,
    'FCF($K)': [round(f, 1) for f in fcf]
})
print("\n=== DCF 5年财务模型 ===")
print(dcf_df.to_string(index=False))

npv = npf.npv(discount_rate, fcf)
print(f"\nNPV @ {discount_rate:.0%} = ${npv:.1f}K")


In [ ]:
# 3. IRR + 回收期 + 盈利指数
irr = npf.irr(fcf)

# 回收期(手动计算, numpy_financial无此函数)
cumulative = 0
payback_period = None
for i, cf in enumerate(fcf):
    cumulative += cf
    if cumulative >= 0 and i > 0:
        prev_cum = cumulative - cf
        payback_period = (i - 1) + (-prev_cum) / cf
        break

# 盈利指数 PI = PV(未来现金流) / |初始投资|
pv_future = sum(fcf[i] / (1 + discount_rate)**i for i in range(1, len(fcf)))
pi = pv_future / abs(fcf[0])

print(f"IRR:                {irr:.2%}")
print(f"Payback Period:     {payback_period:.2f} 年" if payback_period else "Payback Period: >5年")
print(f"Profitability Index: {pi:.2f}")
print(f"投资可行: IRR>{discount_rate:.0%}? {'是' if irr > discount_rate else '否'} | PI>1? {'是' if pi > 1 else '否'}")


## 3. 蒙特卡洛模拟（Monte Carlo Simulation）

DCF 给出 NPV 的点估计，但 AI SaaS 的关键参数高度不确定：
- **Phase 4 ATE**：因果效果估计有置信区间（95% CI: [2.2pp, 5.4pp]）
- **推理成本**：模型 API 价格快速变化（GPT-4 -> DeepSeek 成本降 90%+）
- **客户增长**：市场竞争 + 产品成熟度不确定
- **毛利率**：推理成本曲线决定长期毛利

蒙特卡洛方法：对不确定参数抽样（含ATE） -> 计算每次抽样的 NPV -> 得到估值分布

**scipy.stats / numpy 分布**：
- `np.random.normal(mu, sigma, n)` - 正态分布抽样
- `np.clip(arr, low, high)` - 截断分布范围
- `np.percentile(arr, q)` - 分位数


In [ ]:
# 4. 蒙特卡洛模拟 (10000次) - 传播Phase 4 ATE不确定性
n_sim = 10000

# 参数分布 (Phase 4 ATE置信区间 -> NPV不确定性)
ates_sim = np.clip(np.random.normal(0.038, 0.008, n_sim), 0.010, 0.070)
margins_sim = np.clip(np.random.normal(0.65, 0.05, n_sim), 0.35, 0.85)
growth_mults = np.clip(np.random.normal(1.0, 0.2, n_sim), 0.5, 1.5)
opex_mults = np.random.normal(1.0, 0.15, n_sim)

npv_sim = np.zeros(n_sim)
for i in range(n_sim):
    # ATE -> ARPU 推导
    arpu_k_i = monthly_reach * ates_sim[i] * aov * capture_rate * 12 / 1000
    cust_i = [max(int(c * growth_mults[i]), 1) for c in customers]
    rev_i = [c * arpu_k_i for c in cust_i]
    gp_i = [r * margins_sim[i] for r in rev_i]
    op_i = [o * opex_mults[i] for o in opex]
    cf_i = [gp_i[j] - op_i[j] for j in range(6)]
    cf_i[0] = -initial_investment
    npv_sim[i] = npf.npv(discount_rate, cf_i)

print(f"=== Monte Carlo 估值分布 (n={n_sim}) ===")
print(f"  均值 (Mean NPV):   ${npv_sim.mean():.1f}K")
print(f"  中位数 (Median):   ${np.median(npv_sim):.1f}K")
print(f"  标准差 (Std):      ${npv_sim.std():.1f}K")
print(f"  5%分位 (P5):       ${np.percentile(npv_sim, 5):.1f}K")
print(f"  95%分位 (P95):     ${np.percentile(npv_sim, 95):.1f}K")
print(f"  P(NPV > 0):        {(npv_sim > 0).mean():.1%}")
print(f"\n  Phase 4 ATE不确定性传播: CI[2.2, 5.4]pp -> NPV分布P5-P95跨度")


## 4. 敏感性分析（龙卷风图）

龙卷风图（Tornado Chart）展示各参数对 NPV 的影响排序：
- 对每个参数 +/-20% 变动，计算 NPV 变化范围
- 按影响大小降序排列，形成龙卷风形状
- 识别**高杠杆点**：小投入改变大局的关键参数

**2026前沿 - 推理成本对 AI 估值的影响**：
推理成本是 AI SaaS 估值的核心变量。DeepSeek 等开源模型将推理成本降低 90%+，
直接提升毛利率和估值。本Phase新增 **Phase 4 ATE** 作为敏感性参数，量化因果效果对NPV的影响。


In [ ]:
# 5. 敏感性分析 + 龙卷风图
def calc_npv(ate=0.038, inference_ratio=0.30, data_ratio=0.05,
             growth_mult=1.0, opex_mult=1.0, dr=0.15):
    """计算给定参数下的NPV ($K) - ATE通过ARPU传导"""
    margin = 1.0 - inference_ratio - data_ratio
    arpu_k = monthly_reach * ate * aov * capture_rate * 12 / 1000
    cust = [max(int(c * growth_mult), 1) for c in customers]
    rev = [c * arpu_k for c in cust]
    gp = [r * margin for r in rev]
    op = [o * opex_mult for o in opex]
    cf = [gp[j] - op[j] for j in range(6)]
    cf[0] = -initial_investment
    return npf.npv(dr, cf)

base_npv = calc_npv()

# 各参数+/-20%变动
params_test = {
    'ATE (Phase4)':     lambda d: calc_npv(ate=0.038*(1+d)),
    'Inference Cost':   lambda d: calc_npv(inference_ratio=0.30*(1+d)),
    'Growth':           lambda d: calc_npv(growth_mult=1.0+d),
    'OpEx':             lambda d: calc_npv(opex_mult=1.0+d),
    'Discount Rate':    lambda d: calc_npv(dr=0.15*(1+d)),
}

sensitivity = []
for name, fn in params_test.items():
    npv_high = fn(0.20)
    npv_low = fn(-0.20)
    sensitivity.append({
        '参数': name,
        'Low(-20%)': round(npv_low, 1),
        'Base': round(base_npv, 1),
        'High(+20%)': round(npv_high, 1),
        'Impact': round(abs(npv_high - npv_low), 1)
    })

sensitivity_df = pd.DataFrame(sensitivity).sort_values('Impact', ascending=False)
print("=== 敏感性分析（按影响降序）===")
print(sensitivity_df.to_string(index=False))
print(f"\n最敏感因子: {sensitivity_df.iloc[0]['参数']} (影响: ${sensitivity_df.iloc[0]['Impact']:.1f}K)")

# 龙卷风图
fig, ax = plt.subplots(figsize=(10, 5))
labels = sensitivity_df['参数'].values
lows = sensitivity_df['Low(-20%)'].values
highs = sensitivity_df['High(+20%)'].values
base_val = sensitivity_df['Base'].values[0]
y_pos = range(len(sensitivity_df))
for idx in range(len(sensitivity_df)):
    ax.barh(idx, highs[idx] - base_val, left=base_val, height=0.6, color='steelblue', alpha=0.7)
    ax.barh(idx, lows[idx] - base_val, left=base_val, height=0.6, color='coral', alpha=0.7)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels)
ax.set_xlabel('NPV ($K)')
ax.set_title('敏感性分析 - 龙卷风图 (Capstone Phase 5)')
ax.axvline(x=base_val, color='black', linestyle='--', label=f'Base NPV=${base_val:.0f}K')
ax.legend()
plt.tight_layout()
plt.savefig('tornado_chart.png', dpi=100, bbox_inches='tight')
plt.show()
print("龙卷风图已保存: tornado_chart.png")


## 5. 天道推演 x 投资评估（2026前沿）

> 与项目 CLAUDE.md「天道推演系统」同构。

天道推演是一种元认知沙盘推演能力--以天神视角俯视局势，构建无限可能的沙盘，
模拟不同决策路径下的未来走向。应用于投资评估：

| 天道推演能力 | 投资评估对应 | 实现方式 |
|-------------|------------|---------|
| 局势感知 | 市场环境建模 | 场景定义（含Phase 4 ATE） |
| 因果链追踪 | 价值驱动因素分析 | 敏感性分析 |
| 沙盘模拟（3层） | 多路径推演 | Bull / Base / Bear |
| 概率评估 | 估值概率分布 | 蒙特卡洛模拟 |
| 最优路径推荐 | 投资决策 | NPV / IRR / PI |

**三路径推演**：Bull（乐观, ATE=CI上界5.4pp）/ Base（基准, ATE=3.8pp）/ Bear（悲观, ATE=CI下界2.2pp），每路径推演 3 层（immediate / near / far）。


In [ ]:
# 6. 天道推演多路径场景分析
# Capstone特色: Phase 4 ATE置信区间 -> 天道推演Bull/Bear边界
scenarios = {
    'Bull (乐观)': {'ate': 0.054, 'inference_ratio': 0.23, 'growth_mult': 1.3, 'opex_mult': 0.9, 'dr': 0.12},
    'Base (基准)': {'ate': 0.038, 'inference_ratio': 0.30, 'growth_mult': 1.0, 'opex_mult': 1.0, 'dr': 0.15},
    'Bear (悲观)': {'ate': 0.022, 'inference_ratio': 0.40, 'growth_mult': 0.7, 'opex_mult': 1.2, 'dr': 0.20},
}

scenario_results = []
for name, p in scenarios.items():
    npv_s = calc_npv(**p)

    # 3层推演: immediate(Y1-2) / near(Y3-4) / far(Y5)
    margin = 1.0 - p['inference_ratio'] - 0.05
    arpu_k = monthly_reach * p['ate'] * aov * capture_rate * 12 / 1000
    cust = [max(int(c * p['growth_mult']), 1) for c in customers]
    rev = [c * arpu_k for c in cust]
    gp = [r * margin for r in rev]
    op = [o * p['opex_mult'] for o in opex]
    cf = [gp[j] - op[j] for j in range(6)]
    cf[0] = -initial_investment

    dr = p['dr']
    pv_imm = cf[1]/(1+dr) + cf[2]/(1+dr)**2
    pv_near = cf[3]/(1+dr)**3 + cf[4]/(1+dr)**4
    pv_far = cf[5]/(1+dr)**5

    scenario_results.append({
        '场景': name,
        'NPV($K)': round(npv_s, 1),
        'Immediate(Y1-2)': round(pv_imm, 1),
        'Near(Y3-4)': round(pv_near, 1),
        'Far(Y5)': round(pv_far, 1),
        '可行': '是' if npv_s > 0 else '否'
    })

scenario_df = pd.DataFrame(scenario_results)
print("=== 天道推演：三路径场景分析 ===")
print(scenario_df.to_string(index=False))

# 天道推演风险预警
bull_npv = scenario_results[0]['NPV($K)']
bear_npv = scenario_results[2]['NPV($K)']
print(f"\n=== 天道推演风险预警 ===")
print(f"乐观-悲观跨度: ${bull_npv - bear_npv:.1f}K (不确定性范围)")
print(f"悲观场景NPV: ${bear_npv:.1f}K {'(正, 可承受)' if bear_npv > 0 else '(负, 不可承受)'}")
print(f"关键风险: Phase4 ATE低估 + 推理成本上升 + 客户增长放缓三重打击")
print(f"缓解策略: (1)多模型策略降低推理成本 (2)outcome-based pricing锁定ATE价值")
print(f"          (3)多Agent仿真预判竞争反应 (4)MCP/A2A协议降低生态依赖风险")


## 6. 反思与前沿

### 反思问题
1. MarketingAgent Pro 的 NPV 是多少？IRR 是否高于折现率？投资可行吗？
2. Phase 4 ATE 如何传导为 ARPU 和 NPV？推导链中哪个环节不确定性最大？
3. 蒙特卡洛模拟的 P(NPV>0) 是多少？5% 和 95% 分位差距说明了什么？
4. 敏感性分析中哪个参数对 NPV 影响最大？ATE 排第几？推理成本排第几？
5. 天道推演的三场景中，Bear case 的 NPV 是多少？风险预警是什么？

### 2026前沿：贝叶斯估值（Bayesian Valuation）
传统 DCF 给出点估计 NPV，蒙特卡洛给出频率派分布。**贝叶斯估值**用 PyMC 构建参数的
后验分布，结合先验信息和Phase 4观测数据，给出更稳健的估值后验分布。

### Phase 4-5 整合（Capstone核心）
| Phase | 能力 | Phase 5 整合角色 |
|-------|------|-----------------|
| Phase 4 | 因果实验设计与验证 | ATE -> ARPU -> NPV 推导链 |
| Phase 5 | 商业模式与价值评估 | 画布 + DCF + 蒙特卡洛 + 天道推演 |

### 技能4 Day 1-5 整合
| Day | 能力 | Phase 5 整合角色 |
|-----|------|-----------------|
| Day 1 | AI 商业模式类型学 | 画布的客户细分 + 价值主张 |
| Day 2 | AI 定价策略 | 画布的收入流（outcome-based） |
| Day 3 | Agent 经济学 | 画布的成本结构（推理成本） |
| Day 4 | 平台生态战略 | 画布的核心伙伴 + 渠道 |
| Day 5 | 商业模式画布 + 投资评估 | 画布框架 + NPV/IRR评估 |
